In [7]:
# Initialize Otter
import otter
grader = otter.Notebook("PW4.ipynb")

# **Лабораторная работа: Математический движок Transformer**
## **Численные методы в фундаменте LLM**

В этой работе мы отойдем от простого анализа исторических данных и перейдем к задачам, с которыми сталкиваются инженеры при разработке и оптимизации больших языковых моделей (LLM). Мы изучим, как классическая вычислительная математика позволяет моделям обрабатывать длинные тексты, эффективно сжиматься и аппроксимировать сложные логические функции.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import lagrange, BarycentricInterpolator, CubicSpline
from scipy.optimize import minimize
import otter

---
## **1. Интерполяция: Проблема расширения контекстного окна**

### **Теоретическая справка**
**С точки зрения вычислительной математики:** Интерполяция на равномерно распределенных узлах часто сталкивается с **феноменом Рунге** — резким ростом амплитуды ошибки на краях интервала при увеличении степени полинома. Узлы Чебышева распределены неравномерно (плотнее у краев), что минимизирует максимальное значение ошибки.

**С точки зрения LLM-инженерии:** Модели обучаются на фиксированной длине текста (например, 512 токенов). Чтобы расширить контекст до 2048 без полного переобучения, инженеры используют **Position Interpolation**. Если интерполяция позиционных эмбеддингов дает выбросы на краях, модель «забывает» начало промпта или начинает галлюцинировать в конце.



### **Вопрос 1: Реализация узлов Чебышева**
**Что нужно сделать:**
1. Рассчитайте 10 узлов Чебышева первого рода для стандартного интервала $[-1, 1]$.
2. Масштабируйте их в интервал $[0, 10]$ по формуле: $x_i = \frac{1}{2}(a+b) + \frac{1}{2}(b-a)\cos(\frac{2i-1}{2n}\pi)$.
3. Создайте интерполятор `BarycentricInterpolator` на основе этих узлов для функции $\sin(x)$.

In [11]:
# Синтетические данные эмбеддингов (10 известных точек)
x_known = np.linspace(0, 10, 10)
y_known = np.sin(x_known)


n = 10
a, b = 0, 10
i = np.arange(1, n + 1)

cheb_nodes = 0.5 * (a + b) + 0.5 * (b - a) * np.cos(((2 * i - 1) * np.pi)/(2 * n))
poly_cheb = BarycentricInterpolator(cheb_nodes, np.sin(cheb_nodes))

In [12]:
grader.check("q1")

q1 results: All test cases passed!

### **Вопрос 2: Теоретический анализ феномена Рунге**
Объясните, почему стандартная полиномиальная интерполяция на равномерных узлах терпит неудачу на границах контекстного окна и как узлы Чебышева решают эту проблему.

In [13]:
# Напишите ваш ответ между тройными кавычками ниже.
t2_ans = '''
Равномерные узлы вызывают сильные осцилляции и рост ошибки на границах интервала. Это называется феномен Рунге. Узлы Чебышева решают проблему, сгущаясь к краям интервала и минимизируя максимальную ошибку
'''
# Записываем в файл для автопроверки
with open("t2.txt", "w", encoding="utf-8") as f:
    f.write(t2_ans)

from IPython.display import display, Markdown
display(Markdown(t2_ans))


Равномерные узлы вызывают сильные осцилляции и рост ошибки на границах интервала. Это называется феномен Рунге. Узлы Чебышева решают проблему, сгущаясь к краям интервала и минимизируя максимальную ошибку


In [14]:
grader.check("q2")

q2 results: All test cases passed!

---
## **2. Сплайны: Проектирование функции активации GELU**

### **Теоретическая справка**
**С точки зрения вычислительной математики:** Кубический сплайн — это кусочно-заданная функция, которая сохраняет непрерывность самой функции, её первой и второй производных ($C^2$ непрерывность). Это делает аппроксимацию очень гладкой.

**С точки зрения LLM-инженерии:** Современные модели (BERT, GPT, Llama) используют **GELU (Gaussian Error Linear Unit)**. В отличие от ReLU, GELU дифференцируема во всех точках, что позволяет градиентам течь плавно. Для ускорения работы часто используют табличные значения GELU с последующей сплайн-интерполяцией.



### **Вопрос 3: Расчет производной сплайна**
**Что нужно сделать:**
1. Создайте `CubicSpline` для набора данных `x_nodes` и `y_nodes`.
2. Вычислите градиент (первую производную) в точке $x=0.0$. Это значение критично для предотвращения проблемы «затухающих градиентов».

In [20]:
def gelu_raw(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

x_nodes = np.linspace(-3, 3, 8)
y_nodes = gelu_raw(x_nodes)

cs = CubicSpline(x_nodes, y_nodes)
grad_at_zero = cs(0.0, 1)

In [21]:
grader.check("q3")

q3 results: All test cases passed!

---
## **3. Оптимизация: Нормы L1 и L2 в задачах сжатия моделей**

### **Теоретическая справка**
**С точки зрения вычислительной математики:** L2-норма (квадратичная) минимизирует среднее отклонение и чувствительна к выбросам. L1-норма (модуль) более устойчива к аномалиям и приводит к разреженным решениям (многие веса становятся равными 0).

**С точки зрения LLM-инженерии:** Веса LLM занимают гигабайты памяти. Чтобы запустить модель на смартфоне, мы используем **прунинг (Pruning)**. Мы заставляем оптимизатор использовать L1-регуляризацию, чтобы «обнулить» маловажные связи в нейросети без потери качества. Подробнее про прунинг можно почитать [тут](https://habr.com/ru/articles/811221/).



### **Вопрос 4: Поиск разреженного решения**
**Что нужно сделать:** Реализуйте две функции потерь для поиска параметров прямой $y = ax + b$. В данных есть один гигантский «выброс». Посмотрите, какой метод (L1 или L2) проигнорирует ошибку и сохранит правильный наклон.

In [24]:
weights_x = np.linspace(0, 10, 20)
weights_y = 2 * weights_x + 1
weights_y[-1] = -100 # Мощный шум/выброс

def loss_l2(nums):
    a, b = nums
    return np.sum((a * weights_x + b - weights_y)**2)

def loss_l1(nums):
    a, b = nums
    return np.sum(np.abs(a * weights_x + b - weights_y))

l2_result = minimize(loss_l2, [0, 0])
l1_result = minimize(loss_l1, [0, 0])

l2_slope = l2_result.x[0]
l1_slope = l1_result.x[0]

In [25]:
grader.check("q4")

q4 results: All test cases passed!

### **Вопрос 5: Теория прунинга**
Почему минимизация L1-нормы приводит к появлению нулевых весов (разреженности), а L2 — нет? Как это помогает при запуске LLM на мобильных устройствах?

In [26]:
# Напишите ваш ответ между тройными кавычками ниже.
t5_ans = '''
L1 приводит к появлению нулевых весов, так как её градиент постоянен даже вблизи нуля, в то время как L2 лишь делает делает веса маленькими.
Для мобильных устройств разреженные устройства экономят память и работают быстрее (пропуск умножений на ноль), позволяя запускать тяжёлые LLM на телефонах
'''
# Записываем в файл для автопроверки
with open("t5.txt", "w", encoding="utf-8") as f:
    f.write(t5_ans)

from IPython.display import display, Markdown
display(Markdown(t5_ans))


L1 приводит к появлению нулевых весов, так как её градиент постоянен даже вблизи нуля, в то время как L2 лишь делает делает веса маленькими.
Для мобильных устройств разреженные устройства экономят память и работают быстрее (пропуск умножений на ноль), позволяя запускать тяжёлые LLM на телефонах


In [27]:
grader.check("q5")

q5 results: All test cases passed!

---
## **4. Универсальная аппроксимация: Сигмоиды Цыбенко**

### **Теоретическая справка**
**С точки зрения вычислительной математики:** Теорема Цыбенко утверждает, что любая непрерывная функция на компактном множестве может быть аппроксимирована суммой функций вида $\sum \alpha_i \sigma(w_i x + b_i)$.

**С точки зрения LLM-инженерии:** Это фундаментальное обоснование того, почему нейросети способны «выучить» любую логику, если у них достаточно нейронов. В LLM каждый слой — это, по сути, такая сумма (только в матричном виде).



### **Вопрос 5: Скрытый слой вручную**
**Что нужно сделать:** Напишите код функции, которая вычисляет выход нейронного слоя. Параметры передаются как один плоский массив `params`, который нужно правильно «распаковать» на веса, смещения и коэффициенты суммы.

In [28]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def manual_neural_layer(x, params):
    """
    params: np.array вида [alpha1...alphaN, w1...wN, b1...bN]
    """
    n = len(params) // 3
    alphas = params[0:n]
    weights = params[n:2*n]
    biases = params[2*n:3*n]
    
    return sum(alphas[i] * sigmoid(weights[i] * x + biases[i]) for i in range(n)) 

In [29]:
grader.check("q6")

q6 results: All test cases passed!

---

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [31]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)

/home/artem/ITMO/Семестр 4/ВычМат/Лаба 4/.venv/lib/python3.12/site-packages/otter/check/notebook.py:494: UserWarning: Could not locate a PDF to include
  warnings.warn("Could not locate a PDF to include")


OSError: xelatex not found on PATH, if you have not installed xelatex you may need to do so. Find further instructions at https://nbconvert.readthedocs.io/en/latest/install.html#installing-tex.